# Interactive Visual Analystics with Folium

In [1]:
import folium
import pandas as pd

from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

## Task 1: Mark all the Launching Sites on a map

In [2]:
import requests

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'

In [3]:
def download(filename, url):
    response = requests.get(url)
    if response.status_code == 200:
        with open (filename, 'wb') as f:
            f.write(response.content)

download('spacexfile.csv', URL)

spacex_df = pd.read_csv('spacexfile.csv')
spacex_df.head(5)

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [4]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [5]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [6]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [7]:
site_map = folium.Map(location = nasa_coordinate, zoom_start = 5)

launch_sites = folium.map.FeatureGroup()

for lat, lng, label in zip(launch_sites_df.Lat, launch_sites_df.Long, launch_sites_df['Launch Site']):
    launch_sites.add_child(
       folium.Circle(
           [lat, lng],
           radius = 1000,
           color = '#000000',
           fill = True,
           popup = folium.Popup(label)
       )
    )

for lat, lng, label in zip(launch_sites_df.Lat, launch_sites_df.Long, launch_sites_df['Launch Site']):
    launch_sites.add_child(
        folium.map.Marker(
            [lat, lng],
            icon=DivIcon(icon_size=(20,20),
                         icon_anchor=(0,0),
                         html='<div style="font-size: 12; color:#000000;"><b>%s</b></div>' % label)
            
        )
    )
    
    

site_map.add_child(launch_sites)

## Task 2: Mark the successful/failed launches for each site on the map

In [8]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [9]:
marker_cluster = MarkerCluster()

In [10]:
marker_color = []

for clss in spacex_df['class']:
    if clss == 1:
        marker_color.append('green')
    elif clss == 0:
        marker_color.append('red')

spacex_df['marker_color'] = marker_color
spacex_df.tail(5)

,Launch Site,Lat,Long,class,marker_color
51,CCAFS SLC-40,28.563197,-80.57682,0,red
52,CCAFS SLC-40,28.563197,-80.57682,0,red
53,CCAFS SLC-40,28.563197,-80.57682,0,red
54,CCAFS SLC-40,28.563197,-80.57682,1,green
55,CCAFS SLC-40,28.563197,-80.57682,0,red


In [11]:
site_map.add_child(marker_cluster)

for lat, lng, marker_color in zip(spacex_df.Lat, spacex_df.Long, spacex_df.marker_color):
    marker = folium.Marker(
        [lat, lng],
        icon = folium.Icon(color = 'white', icon_color = marker_color)
    )
    marker_cluster.add_child(marker)

site_map

## Task 3: Calculate the distances between a launch site to its proximities

In [12]:
## add mouse position to get the coordinate (Lat, Long) for a mouse over the map

formatter = 'function(num) {return L.Util.formatNum(num, 5); };'

mouse_position = MousePosition(
    position = 'topright',
    separator = 'Long',
    empty_string = 'NaN',
    lng_first = False, 
    num_digits = 20,
    prefix = 'Lat: ',
    lat_formatter = formatter, 
    lng_formatter = formatter
)

site_map.add_child(mouse_position)
site_map

In [13]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [14]:
Lat = 34.63931
Long = -120.63194

distance_coastline = calculate_distance(34.632834, -120.610745, Lat, Long)
distance_coastline

2.0691443829372203

In [19]:
distance_marker = folium.Marker(
    [Lat, Long],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )

site_map.add_child(distance_marker)


In [21]:
lines = folium.PolyLine(locations = [[34.632834, -120.610745], [Lat, Long]], weight=1)

site_map.add_child(lines)